# W16-D2 实验：G-05 Effect-Registry 代码锚点 × Frozen CI 红绿验证

**与 md 的分工**：md（`第16周-Day2-G05EffectRegistry代码锚点与FrozenCI.md`）是阅读材料；本 ipynb 是**可执行证据**——真实跑通 15 锚点对账、复现 8 场景红绿判决矩阵、并用蒙特卡洛对比三种锚点策略在代码演化下的告警行为。

**今日问题**：`# Registry frozen v1.0` 只是 effect-registry.yaml 里的一行注释——当它被违反时，谁会说"不"？

工件：`semantic-model/governance/g05-effect-anchors.yaml`（15 锚点登记）+ `semantic-model/governance/ci/frozen_effect_ci.py`（两门 CI 脚本）。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## §1 登记面：5 类冻结 effect × 15 锚点

effect-registry v1.0（2026-07-26 冻结）声明 5 类 Lifecycle Effect 的**存在性与意图**；今天补上机器可读的**执行证据锚点**。每类 3 锚，覆盖三个侧面：声明侧（状态字面量/类型定义）、执行侧（写入/生成入口）、档案侧（审计/测试）——单一侧面的证据都会腐烂或说谎。

In [ ]:
import subprocess, json, yaml, tempfile, shutil, random
from pathlib import Path

GOV = Path("/root/learning-notebooks/semantic-model/governance")
CI = str(GOV / "ci" / "frozen_effect_ci.py")
ANCH = str(GOV / "g05-effect-anchors.yaml")
REG = Path("/root/docs/lanlnk/config/ontology/cre-business-capability-matrix/effect-registry.yaml")
LNKCRE = Path("/root/lnkcre")
OUTDIR = Path("/root/learning-notebooks/第16周")

anchors_doc = yaml.safe_load(Path(ANCH).read_text(encoding="utf-8"))
registry = yaml.safe_load(REG.read_text(encoding="utf-8"))

frozen_ids = [e["id"] for e in registry["effect_types"]]
counts = {e: len(s["anchors"]) for e, s in anchors_doc["effects"].items()}
print("冻结 effect 类型（registry 声明侧）:", frozen_ids)
print("锚点登记（每类 3 锚 = 声明/执行/档案三侧面）:", counts)
print("基线 commit:", anchors_doc["baseline_commit"][:9], " 验证日:", anchors_doc["verified_date"])
n_anchor = sum(len(s["anchors"]) for s in anchors_doc["effects"].values())
assert set(anchors_doc["effects"]) == set(frozen_ids), "锚点登记与冻结清单必须一一对应"
print("登记完整性:", len(frozen_ids), "/", len(frozen_ids), "类 × 3 =", n_anchor, "锚点")

## §2 真实门禁：15 锚点对当前 lnkcre 全量对账

调 CI 脚本 `anchors` 门（子进程执行，与 CLI 生产路径完全一致），期望 15/15 OK。

In [ ]:
r = subprocess.run(["python3", CI, "anchors", "--anchors", ANCH, "--repo", str(LNKCRE), "--json"],
                   capture_output=True, text=True)
out = json.loads(r.stdout)
print("anchors 门 exit =", r.returncode, "（0=绿）")
for row in out["results"]:
    mark = {"OK": "OK ", "DRIFTED": "DRIFT", "BROKEN": "BREAK"}[row["verdict"]]
    print(mark, row["effect"].ljust(24), row["file"].split("/")[-1] + ":" + str(row["line"]))
print("汇总:", out["summary"])
assert r.returncode == 0 and out["summary"]["ok"] == 15, "真实基线必须 15/15 全绿"
print("真实基线验证通过：15/15 OK")

## §3 红绿判决矩阵：8 场景实测（brief 验收②证据）

在临时仓副本上复现四类锚点场景 + 四类 registry 场景。关键设计：**BROKEN（证据被改）才红；DRIFTED（位置漂移）只黄**——上游插行不是治理违规，是维护信号。

In [ ]:
TMPD = []  # 收集临时目录，末尾统一清理

def make_demo_repo():
    d = Path(tempfile.mkdtemp()); TMPD.append(d)
    for s in anchors_doc["effects"].values():
        for a in s["anchors"]:
            dst = d / a["file"]; dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(LNKCRE / a["file"], dst)
    return d

def run_anchors(repo, *extra, changes_dir=None):
    cmd = ["python3", CI, "anchors", "--anchors", ANCH, "--repo", str(repo), "--json"]
    if changes_dir is not None:
        cmd += ["--changes-dir", str(changes_dir)]
    r = subprocess.run(cmd + list(extra), capture_output=True, text=True)
    return r.returncode, json.loads(r.stdout)["summary"]

def run_registry(new_yaml, *extra):
    changes = Path(tempfile.mkdtemp()); TMPD.append(changes)
    (changes / "g05-demo").mkdir(); (changes / "g05-demo" / "proposal.md").write_text("# demo")
    r = subprocess.run(["python3", CI, "registry", "--old", str(REG), "--new", str(new_yaml),
                        "--changes-dir", str(changes), "--json", *extra], capture_output=True, text=True)
    return r.returncode, json.loads(r.stdout)

scenarios = []

rc, s = run_anchors(LNKCRE)
scenarios.append(("S1 真实基线 15 锚", "anchors", rc, "OK %d/15" % s["ok"]))

repo = make_demo_repo()
p = repo / "backend/internal/lease/model.go"
p.write_text("// churn\n" * 3 + p.read_text())
rc, s = run_anchors(repo)
scenarios.append(("S2 上游插行×3", "anchors", rc, "DRIFTED %d（黄，维护信号）" % s["drifted"]))

repo = make_demo_repo()
p = repo / "backend/internal/occupancy/repository.go"
p.write_text(p.read_text().replace("func (r *Repository) CreateTx(", "func (r *Repository) CreateTxLegacy("))
rc, s = run_anchors(repo)
scenarios.append(("S3 篡改实现证据", "anchors", rc, "BROKEN %d（红）" % s["broken"]))

repo = make_demo_repo()
p = repo / "backend/internal/occupancy/repository.go"
p.write_text(p.read_text().replace("func (r *Repository) CreateTx(", "func (r *Repository) CreateTxLegacy("))
chg = Path(tempfile.mkdtemp()); TMPD.append(chg)
(chg / "g05-demo").mkdir(); (chg / "g05-demo" / "proposal.md").write_text("# demo")
rc, s = run_anchors(repo, "--change", "g05-demo", changes_dir=chg)
scenarios.append(("S4 篡改+立案change", "anchors", rc, "BROKEN 被立案承认 → 不红"))

rc, o = run_registry(REG)
scenarios.append(("S5 registry 无变更", "registry", rc, "no diff"))

def tampered_registry():
    d = yaml.safe_load(REG.read_text(encoding="utf-8"))
    d["effect_types"].append({"id": "service-effect", "description": "未过 ORE-1 的第6类",
                              "owner_domain": "service", "status": "published"})
    f = Path(tempfile.mkdtemp()) / "t.yaml"; TMPD.append(f.parent)
    f.write_text(yaml.safe_dump(d, allow_unicode=True))
    return f

t = tampered_registry()
rc, o = run_registry(t); scenarios.append(("S6 新增第6类(无change)", "registry", rc, "added: service-effect"))
rc, o = run_registry(t, "--change", "g05-demo"); scenarios.append(("S7 新增+立案change", "registry", rc, "合法变更（ORE-1 逃逸口）"))
rc, o = run_registry(t, "--change", "g99-not-filed"); scenarios.append(("S8 change未立案", "registry", rc, "立案是逃逸口不是注释"))

print("%-24s %-10s %-6s %s" % ("场景", "门", "exit", "判决"))
for name, gate, rc, note in scenarios:
    print("%-24s %-10s %-6d %s" % (name, gate, rc, note))

expect_red = {"S3 篡改实现证据", "S6 新增第6类(无change)", "S8 change未立案"}
for name, gate, rc, note in scenarios:
    assert (rc == 1) == (name in expect_red), name
print()
print("断言通过：红绿分布与设计完全一致（3 红 5 绿）")

In [ ]:
# 图1：红绿判决矩阵
from matplotlib.patches import Rectangle
GREEN, RED = "#2e7d32", "#c62828"
fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, gate, rc, note) in enumerate(scenarios):
    ax.add_patch(Rectangle((0, i - 0.45), 1, 0.9, color=GREEN if rc == 0 else RED))
    ax.text(0.015, i, name, ha="left", va="center", color="white", fontsize=11, fontweight="bold")
    ax.text(0.42, i, "[" + gate + "]", ha="left", va="center", color="white", fontsize=8.5)
    ax.text(0.55, i, note, ha="left", va="center", color="white", fontsize=9)
    ax.text(0.975, i, "exit=%d" % rc, ha="right", va="center", color="white", fontsize=11, fontweight="bold")
ax.set_xlim(0, 1); ax.set_ylim(len(scenarios) - 0.5, -0.5)
ax.axis("off")
ax.set_title("G-05 frozen CI 红绿判决矩阵：改冻结项不带 change 即红，带立案 change 才绿", fontsize=12.5)
fig.tight_layout()
fig.savefig(OUTDIR / "w16d2_gate_matrix.png", dpi=150); plt.close()
print("已保存 w16d2_gate_matrix.png")

## §4 锚点腐化蒙特卡洛：三种锚点策略谁先撒谎

lnkcre 演化速率 ~15 commits/天，**行号必然腐烂**。用两个真实锚点文件做 4 场景 × 400 次模拟，对比：
- **策略A 包路径锚点**（W14-D4 现状：registry 只说"在 lease 包"）——从不告警；
- **策略B 纯行号**——行内容变了就红，不区分"上游插行"与"证据被改"；
- **策略C 行号+内容（本设计）**——证据消失才红，位置漂移只黄。

另含一个诚实场景：**扩名不改名**（`StatusDraft → StatusDraftLegacy`）——子串匹配仍命中，C 会漏报。这是子串锚点的真实盲区，量化出来才知道边界在哪。

In [ ]:
random.seed(42)
# (文件, 行, expect 子串, 改名后) —— 改名后的新名不得包含原子串
ANCHOR_TARGETS = [("backend/internal/lease/model.go", 16, "StatusDraft", "LeaseDraftStatus"),
                  ("backend/internal/occupancy/repository.go", 26,
                   "func (r *Repository) CreateTx(", "func (r *Repository) CreateTxV2(")]
SCEN = ["benign_above", "benign_below", "tamper_in_place", "tamper_extend", "delete_line"]
N = 200

def strategy_A(new_lines, L, exp):  # 包路径锚点：只登记"大概在哪"，永远沉默
    return "SILENT"

def strategy_B(new_lines, L, exp, orig_line):  # 纯行号：登记行内容变了就红
    if L > len(new_lines) or new_lines[L - 1] != orig_line:
        return "RED"
    return "GREEN"

def strategy_C(new_lines, L, exp, orig_line):  # 行号+内容：证据消失红，位置漂移黄
    if 1 <= L <= len(new_lines) and exp in new_lines[L - 1]:
        return "GREEN"
    if any(exp in l for l in new_lines):
        return "AMBER"
    return "RED"

stats = {s: {"A": 0, "B": 0, "C": 0, "C_amber": 0} for s in SCEN}
for rel, L, exp, exp_new in ANCHOR_TARGETS:
    orig_lines = (LNKCRE / rel).read_text().splitlines()
    orig_line = orig_lines[L - 1]
    for scen in SCEN:
        for _ in range(N):
            k = random.randint(1, 10)
            churn = ["// churn %.6f" % random.random() for _ in range(k)]
            ls = list(orig_lines)
            if scen == "benign_above":
                ls = churn + ls
            elif scen == "benign_below":
                ls = ls[:L] + churn + ls[L:]
            elif scen == "tamper_in_place":   # 改名：新名不含旧子串 → 证据消失
                ls[L - 1] = ls[L - 1].replace(exp, exp_new)
            elif scen == "tamper_extend":      # 扩名不改名：旧子串仍被包含 → 子串锚点盲区
                ls[L - 1] = ls[L - 1].replace(exp, exp + "Legacy")
            elif scen == "delete_line":
                del ls[L - 1]
            if strategy_A(ls, L, exp) == "RED":
                stats[scen]["A"] += 1
            if strategy_B(ls, L, exp, orig_line) == "RED":
                stats[scen]["B"] += 1
            v = strategy_C(ls, L, exp, orig_line)
            if v == "RED":
                stats[scen]["C"] += 1
            elif v == "AMBER":
                stats[scen]["C_amber"] += 1

per_anchor = N * len(ANCHOR_TARGETS)
zh = {"benign_above": "上游插行(良性)", "benign_below": "下游插行(良性)",
      "tamper_in_place": "改名(违规)", "tamper_extend": "扩名不改名(违规)", "delete_line": "删除锚点行(违规)"}
print("%-18s %9s %9s %10s   %s" % ("场景", "A包路径", "B纯行号", "C行+内容", "C黄(维护)"))
for s in SCEN:
    print("%-16s %8.0f%% %8.0f%% %9.0f%%   %7.0f%%" % (zh[s],
          100 * stats[s]["A"] / per_anchor, 100 * stats[s]["B"] / per_anchor,
          100 * stats[s]["C"] / per_anchor, 100 * stats[s]["C_amber"] / per_anchor))
assert stats["benign_above"]["C"] == 0 and stats["benign_below"]["C"] == 0, "C 良性场景必须 0 误报"
assert stats["tamper_in_place"]["C"] == per_anchor and stats["delete_line"]["C"] == per_anchor, "C 对改名/删除必须 100% 抓获"
assert stats["tamper_extend"]["C"] == 0, "C 对扩名不改名盲（已量化登记的盲区，非事故）"
assert stats["tamper_in_place"]["A"] == 0, "策略A 对违规静默放行（现状缺口）"
assert stats["benign_above"]["B"] == per_anchor, "策略B 良性场景 100% 误报（狼来了）"
print()
print("断言通过：C 0 误报 / 改名删除 100% 抓获 / 扩名盲区已显式登记；A 静默腐烂；B 狼来了")

In [ ]:
# 图2：三策略判红率对比（400 次/场景）
import numpy as np
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(SCEN)); w = 0.25
series = [("A", "A 包路径（W14-D4 现状）", "#9e9e9e"),
          ("B", "B 纯行号", "#f9a825"),
          ("C", "C 行号+内容（本设计）", "#1565c0")]
for j, (key, label, c) in enumerate(series):
    vals = [stats[s][key] / per_anchor for s in SCEN]
    bars = ax.bar(x + (j - 1) * w, vals, w, label=label, color=c)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + w / 2, v + 0.02, "%.0f%%" % (100 * v), ha="center", fontsize=8)
vals_amber = [stats[s]["C_amber"] / per_anchor for s in SCEN]
ax.bar(x + w, vals_amber, w * 0.4, color="#1565c0", alpha=0.35, hatch="//", label="C 黄(漂移=维护信号)")
ax.axhline(1.0, ls="--", lw=0.8, color="#c62828")
ax.set_xticks(x); ax.set_xticklabels([zh[s] for s in SCEN])
ax.set_ylabel("判红率（exit 1）"); ax.set_ylim(0, 1.18)
ax.set_title("锚点腐化模拟（5 场景 × 400 次，真实文件）：A 静默腐烂 · B 狼来了 · C 精准但有扩名盲区", fontsize=12)
ax.legend(loc="upper left", fontsize=9)
ax.text(3.45, 1.03, "虚线=违规场景理想判红率 100%", color="#c62828", fontsize=8, ha="right")
fig.tight_layout()
fig.savefig(OUTDIR / "w16d2_anchor_strategies.png", dpi=150); plt.close()
print("已保存 w16d2_anchor_strategies.png")

## §5 结论

1. **frozen 的执行者**：从"读注释的人"变成"无 change 即红的退出码"——brief 验收②（红绿可测）在 8 场景下双向成立：篡改/未立案均 exit 1，带已立案 change 才 exit 0。
2. **锚点会腐化，设计要承认**：15 commits/天 下行号必然漂移。C 策略（file+line+expect 三元组）把"证据消失"（红，违规）与"位置漂移"（黄，维护）分开——良性场景 0 误报、改名/删除 100% 抓获；**但对"扩名不改名"（StatusDraft→StatusDraftLegacy）存在已量化的盲区**——子串锚点不是终点，需与 registry 门、周期指纹复核（D1 模式）配合成网。
3. **两个 W14-D4 缺口同日关闭**：缺口1（registry 无机器锚点）→ 15 锚点登记全绿；缺口2（frozen 无 CI 强制）→ registry 门。与 D1 的 ontology 指纹链是同一治理模式：**不防变化，让变化留下机器可核对的痕迹**。

In [ ]:
print("=" * 64)
print("W16-D2 验收对照（w16-dev-brief 工作项②：G-05 锚点 + frozen CI）")
print("=" * 64)
print("改 frozen effect 不带 change → CI 红 : OK  S3/S6/S8 实测 exit=1")
print("带 change → 绿（本地可演示）      : OK  S4/S7 实测 exit=0（须已立案）")
print("15 锚点真实基线                   : OK  15/15 @ 0392e107（2026-09-15）")
print("锚点盲区（扩名不改名）             : 已量化登记（ipynb §4），配 registry 门+指纹复核")
print("W14-D4 缺口1（registry 无机器锚点）    : 关闭")
print("W14-D4 缺口2（frozen 无 CI 强制）     : 关闭")
print("=" * 64)
print("产物：semantic-model/governance/g05-effect-anchors.yaml")
print("      semantic-model/governance/ci/frozen_effect_ci.py")
print("      w16d2_gate_matrix.png / w16d2_anchor_strategies.png")
for d in TMPD:
    shutil.rmtree(d, ignore_errors=True)
print("临时演示目录已清理:", len(TMPD))